# Animating a numpy simulation: reaction-diffusion

In [ ]:
%load_ext mcidasv_jupyter
%mcv_connect /path/to/runMcV
%mcv_replay off

## 1. Simulate Gray-Scott in numpy

In [ ]:
import numpy as np

H, W = 140, 220
U = np.ones((H, W), 'f8'); V = np.zeros((H, W), 'f8')
r = 12
cy, cx = H // 2, W // 2
U[cy-r:cy+r, cx-r:cx+r] = 0.50
V[cy-r:cy+r, cx-r:cx+r] = 0.25
V += 0.02 * np.random.default_rng(0).random((H, W))

Du, Dv, feed, kill = 0.16, 0.08, 0.060, 0.062
def lap(Z):
    return (np.roll(Z,1,0)+np.roll(Z,-1,0)+np.roll(Z,1,1)+np.roll(Z,-1,1) - 4*Z)

frames = []
for step in range(6000):
    uvv = U * V * V
    U += Du*lap(U) - uvv + feed*(1-U)
    V += Dv*lap(V) + uvv - (feed+kill)*V
    if step % 750 == 0:
        frames.append(V.copy())
cube = np.stack(frames).astype('f4')
print('frames:', cube.shape)

## 2. Preview a couple of frames

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, idx in zip(ax, [len(cube)//4, len(cube)//2, -1]):
    a.imshow(cube[idx], cmap='magma'); a.axis('off'); a.set_title('frame %d' % (idx % len(cube)))
plt.tight_layout()

## 3. Loop it in McIDAS-V and save a movie

In [ ]:
import os, mcidasv_jupyter as mcv
session = mcv.get_session()
lats = np.linspace(20, 52, cube.shape[1]); lons = np.linspace(-125, -66, cube.shape[2])

OUTDIR = 'output'
os.makedirs(OUTDIR, exist_ok=True)
movie = os.path.join(OUTDIR, 'reaction_diffusion.gif')
session.animate_grid(cube, lats, lons, name='V', out=movie, fps=4,
                     setup="panel[0].annotate('Gray-Scott reaction-diffusion (numpy)', line=470, element=380, size=16, color='white')")